In [2]:
start_month = "2025-01"
end_month = "2025-06"

StatementMeta(, abaf8261-49e2-482b-b92a-57b041110011, 5, Finished, Available, Finished, False)

In [3]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window


df = spark.read.table("openaq_sensor_days")
df.printSchema()
df.show(10)

df.select("date_local").distinct().show(5, truncate=False)
df.groupBy("coverage_pct").count().orderBy("coverage_pct").show(50)

df.groupBy("parameter", "units").count().show()

df.filter(df.avg_value < 0).count()


StatementMeta(, abaf8261-49e2-482b-b92a-57b041110011, 6, Finished, Available, Finished, False)

root
 |-- is_monitor: boolean (nullable = true)
 |-- location_id: long (nullable = true)
 |-- sensor_id: long (nullable = true)
 |-- parameter: string (nullable = true)
 |-- units: string (nullable = true)
 |-- avg_value: double (nullable = true)
 |-- date_local: string (nullable = true)
 |-- coverage_pct: double (nullable = true)

+----------+-----------+---------+---------+-----+---------+--------------------+------------+
|is_monitor|location_id|sensor_id|parameter|units|avg_value|          date_local|coverage_pct|
+----------+-----------+---------+---------+-----+---------+--------------------+------------+
|      true|        664|     1145|     pm25|µg/m³|      3.3|2/18/2025 12:00:0...|        42.0|
|      true|        664|     1145|     pm25|µg/m³|     11.6|5/29/2025 12:00:0...|        42.0|
|      true|        664|     1145|     pm25|µg/m³|     10.9|6/25/2025 12:00:0...|        42.0|
|      true|     387315|  2165515|     pm25|µg/m³|     3.77|3/26/2025 12:00:0...|        42.0|
|

0

In [4]:
df = df.withColumn("measure_date", F.to_date(F.split(F.col("date_local"), " ").getItem(0), "M/d/yyyy"))

null_dates = df.filter(F.col("measure_date").isNull()).count()
if null_dates > 0:
    raise ValueError(f"{null_dates} rows have a date that could not be parsed")
    
# only the requested period
df = df.filter(
    (F.date_format("measure_date", "yyyy-MM") >= start_month) &
    (F.date_format("measure_date", "yyyy-MM") <= end_month)
)

df = df.withColumn("is_low_coverage", F.col("coverage_pct") < 75)

df_s = spark.read.table("openaq_nyc_sensors").select("sensor_id", "location_name", "latitude", "longitude")
df_res = df.join(df_s, "sensor_id", "left")

df_res = df_res.select(
    "measure_date", "sensor_id", "location_id", "location_name",
    "latitude", "longitude", "is_monitor", "parameter",
    "units", "avg_value", "coverage_pct", "is_low_coverage"
)

print("rows:", df_res.count())
print("low coverage:", df_res.filter(F.col("is_low_coverage")).count())

df_res.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("lh_silver.dbo.air_quality_daily")

StatementMeta(, abaf8261-49e2-482b-b92a-57b041110011, 7, Finished, Available, Finished, False)

rows: 4744
low coverage: 138


In [5]:
spark.read.table("openaq_nyc_sensors") \
    .select("location_id", "location_name", "latitude", "longitude") \
    .distinct().orderBy("location_name").show(50, truncate=False)

StatementMeta(, abaf8261-49e2-482b-b92a-57b041110011, 8, Finished, Available, Finished, False)

+-----------+---------------------------------------------+------------------+------------------+
|location_id|location_name                                |latitude          |longitude         |
+-----------+---------------------------------------------+------------------+------------------+
|2616564    |7th Ave and W 16th St                        |40.740022         |-73.998504        |
|1496       |Bayonne                                      |40.67025          |-74.126083        |
|1824516    |Bayside, NY                                  |40.764348694977386|-73.76633985508255|
|3406357    |Bayside, NY 11361                            |40.76433675598798 |-73.76633802306293|
|648        |Bklyn - PS 314                               |40.641819         |-74.018707        |
|664        |Bklyn - PS274                                |40.694401         |-73.928596        |
|626        |Bronx - IS52                                 |40.816101         |-73.902199        |
|665        |Bronx -

In [6]:
locations = [
    (2616564, "Manhattan", True),
    (384,     "Manhattan", True),
    (3400913, "Manhattan", True),
    (4727343, "Manhattan", True),
    (625,     "Manhattan", True),
    (648,     "Brooklyn", True),
    (664,     "Brooklyn", True),
    (3041962, "Brooklyn", True),
    (2903996, "Brooklyn", True),
    (626,     "Bronx", True),
    (665,     "Bronx", True),
    (292229,  "Bronx", True),
    (1824516, "Queens", True),
    (3406357, "Queens", True),
    (3181018, "Queens", True),
    (628,     "Queens", True),
    (631,     "Queens", True),
    (8749,    "Queens", True),
    (496096,  "Staten Island", True),
    (1496,    "New Jersey", False),
    (2386747, "New Jersey", False),
    (3305066, "New Jersey", False),
    (971,     "New Jersey", False),
    (857,     "New Jersey", False),
    (1122,    "New Jersey", False),
    (928,     "New Jersey", False),
    (2153518, "New Jersey", False),
    (3103622, "New Jersey", False),
    (4419380, "New Jersey", False),
    (391818,  "New Jersey", False),
    (387315,  "New Jersey", False),
]

df_loc = spark.createDataFrame(locations, ["location_id", "borough", "in_nyc"])

df_loc.groupBy("borough", "in_nyc").count().orderBy("borough").show()

df_loc.write.mode("overwrite").format("delta").saveAsTable("lh_silver.dbo.aq_location_ref")

StatementMeta(, abaf8261-49e2-482b-b92a-57b041110011, 9, Finished, Available, Finished, False)

+-------------+------+-----+
|      borough|in_nyc|count|
+-------------+------+-----+
|        Bronx|  true|    3|
|     Brooklyn|  true|    4|
|    Manhattan|  true|    5|
|   New Jersey| false|   12|
|       Queens|  true|    6|
|Staten Island|  true|    1|
+-------------+------+-----+



In [7]:
aq = spark.read.table("lh_silver.dbo.air_quality_daily")

joined = aq.join(df_loc, "location_id", "left")

print("rows without borough:", joined.filter(F.col("borough").isNull()).count())

StatementMeta(, abaf8261-49e2-482b-b92a-57b041110011, 10, Finished, Available, Finished, False)

rows without borough: 0
